In [1]:
import pandas as pd
import sqlite3

## create a connection to the database using the library sqlite3

In [2]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

## get the schema of the table test

In [3]:
schema_of_test = pd.read_sql('PRAGMA table_info(test)', conn)
schema_of_test

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


## get only the first 10 rows of the table test

In [4]:
test_10 = pd.read_sql('SELECT * FROM test LIMIT 10', conn)
test_10

,uid,labname,first_commit_ts,first_view_ts
0,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
1,user_30,laba04,2020-04-18 13:36:53.971502,2020-04-17 22:46:26.785035
2,user_30,laba04s,2020-04-18 14:51:37.498399,2020-04-17 22:46:26.785035
3,user_14,laba04,2020-04-18 15:14:00.312338,2020-04-18 10:53:52.623447
4,user_14,laba04s,2020-04-18 22:30:30.247628,2020-04-18 10:53:52.623447
5,user_19,laba04,2020-04-20 19:05:01.297780,2020-04-21 20:30:38.034966
6,user_25,laba04,2020-04-20 19:16:50.673054,2020-05-09 23:54:54.260791
7,user_21,laba04,2020-04-21 17:48:00.487806,2020-04-22 22:40:36.824081
8,user_30,project1,2020-04-22 12:36:24.053518,2020-04-17 22:46:26.785035
9,user_21,laba04s,2020-04-22 20:09:21.857747,2020-04-22 22:40:36.824081


## find among all the users the minimum value of the delta detween the first commit of the user and the deadline of the corresponding lab using one query

In [5]:
deadline = pd.read_sql('SELECT * FROM deadlines LIMIT 10', conn)
deadline

,index,labs,deadlines
0,0,laba04,1587945599
1,1,laba04s,1587945599
2,2,laba05,1588550399
3,4,laba06,1590364799
4,5,laba06s,1590364799
5,3,project1,1589673599


In [6]:
schema_of_deadlines = pd.read_sql('''
SELECT strftime('%s', first_commit_ts) AS unix_time
FROM test
''', conn)
schema_of_deadlines.head()

,unix_time
0,1587196605
1,1587217013
2,1587221497
3,1587222840
4,1587249030


In [7]:
df_min = pd.read_sql('''
SELECT t.uid, MIN((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600) AS MIN_diff
FROM test t JOIN deadlines d
    ON t.labname = d.labs
WHERE d.labs <> 'project1'
''', conn)

df_min

,uid,MIN_diff
0,user_30,-202


## do the same thing, but for the maximum, using only one query, the dataframe name is df_max

In [8]:
df_max = pd.read_sql('''
SELECT t.uid, MAX((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600) AS MAX_diff
FROM test t JOIN deadlines d
    ON t.labname = d.labs
WHERE d.labs <> 'project1'
''', conn)

df_max

,uid,MAX_diff
0,user_25,-2


## do the same thing but for the average

In [9]:
df_avg = pd.read_sql('''
SELECT AVG((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600) AS AVG_diff
FROM test t JOIN deadlines d
    ON t.labname = d.labs
WHERE d.labs <> 'project1'
''', conn)

df_avg

,AVG_diff
0,-89.125


## calculate the correlation corffcient between the number of pageviews and the difference

In [10]:
views_diff = pd.read_sql('''
SELECT t.uid,
    AVG((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600) AS avg_diff,
    pageviews
FROM 
    test t 
    JOIN deadlines d ON t.labname = d.labs
    JOIN 
    (
        SELECT uid, COUNT(datetime) AS pageviews
        FROM pageviews
        GROUP BY uid
    ) as p ON t.uid = p.uid
WHERE t.labname <> 'project1' 
GROUP BY t.uid
''', conn)

views_diff

,uid,avg_diff,pageviews
0,user_1,-64.400000,28
1,user_10,-74.800000,89
2,user_14,-159.000000,143
3,user_17,-61.600000,47
4,user_18,-5.666667,3
5,user_19,-98.750000,16
6,user_21,-95.500000,10
7,user_25,-92.600000,179
8,user_28,-86.400000,149
9,user_3,-105.400000,317


In [11]:
views_diff.corr(numeric_only=True)

,avg_diff,pageviews
avg_diff,1.000000,-0.279736
pageviews,-0.279736,1.000000


In [12]:
views_per_user = pd.read_sql('''
SELECT uid, COUNT(datetime)
FROM pageviews
GROUP BY uid
''', conn)
views_per_user

,uid,COUNT(datetime)
0,admin_0,3
1,admin_1,71
2,admin_2,1
3,admin_3,18
4,user_1,28
5,user_10,89
6,user_14,143
7,user_17,47
8,user_18,3
9,user_19,16


## close the connection

In [13]:
conn.close()